# Kaggle Training Notebook
## Install Dependencies

In [1]:
# install Python package dependencies
!pip install torch-geometric
!pip install -q huggingface_hub

from IPython.display import clear_output
clear_output()

In [2]:
# download or update ll-hls4ml (development library)
import os, sys
if os.path.isdir("ll-hls4ml/.git"):
    !cd ll-hls4ml && git pull --rebase
else:
    !git clone https://github.com/brios-polimi/ll-hls4ml.git
sys.path.append("/kaggle/working/ll-hls4ml/src")

# import importlib
# import ll_hls4ml.training.loops
# importlib.reload(ll_hls4ml.training.loops)

Cloning into 'll-hls4ml'...
remote: Enumerating objects: 231, done.
remote: Counting objects: 100% (231/231), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 231 (delta 107), reused 206 (delta 82), pack-reused 0 (from 0)
Receiving objects: 100% (231/231), 9.45 MiB | 30.13 MiB/s, done.
Resolving deltas: 100% (107/107), done.


In [3]:
import torch
import torch.nn as nn

from ll_hls4ml.data.dataset import HeteroGraphDataset
from ll_hls4ml.data.vocab import load_vocab
from ll_hls4ml.models.registry import build
from ll_hls4ml.training.loops import fit, _json_converter

In [4]:
# download graph tensors dataset from HuggingFace
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

from huggingface_hub import snapshot_download, login
login(hf_token)
tensor_path = snapshot_download(repo_id="BrendanRios/wa-hls4ml-tensors", repo_type="dataset")

Fetching ... files: 0it [00:00, ?it/s]

## Data Loading

In [5]:
max_per_type_train_val = {
    "2layer":         300,
    "3layer":         300,
    "conv1d":         300,
    "conv2d":         300,
    "dense_latency":  300,
    "dense_resource": 300,
    "exemplar":       300,
    "rule4ml":        300
}
max_per_type_test = {
    #"2layer":         100,
    #"3layer":         100,
    #"conv1d":         100,
    #"conv2d":         100,
    #"dense_latency":  100,
    #"dense_resource": 100,
    #"exemplar":       100,
    #"rule4ml":        100
}
train_val_kernel_types = list[str](max_per_type_train_val.keys())
test_kernel_types = list[str](max_per_type_test.keys())
vocab, max_pos, _ = load_vocab(tensor_path + "/vocab.json")

In [6]:
# train_val_ds = HeteroGraphDataset(tensor_path, types=train_val_kernel_types, max_per_type=300, silent=False)
# test_ds = HeteroGraphDataset(cfg.tensor_dir, types=test_kernel_types, max_per_type=max_per_type_test, silent=False)
# train_ds, val_ds = random_train_val_split(train_val_ds, val_fraction=0.2)

## Model Creation

In [7]:
EPOCHS     = 200
BATCH_SIZE = 8
PATIENCE   = 25
VERBOSITY  = 2
SEED       = 42

torch.manual_seed(SEED)

HIDDEN_DIM = 32 # node hidden dimensions
NUM_LAYERS = 2  # number GNN layers
DROPOUT    = 0.3 # node dropout for every GNN layer

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocab, max_pos, _ = load_vocab(tensor_path + "/vocab.json")
vocab_sizes = {k: len(v) for k, v in vocab.items()}

model = build(
    "rgcn",
    node_vocab_sizes=vocab_sizes,
    edge_pos_vocab_size=max_pos,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
)#.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# train_loader = make_loader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
# val_loader = make_loader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
# test_loader = make_loader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

/kaggle/working/ll-hls4ml/src/ll_hls4ml/models/rgcn.py:45: UserWarning: There exist node types ({'constant'}) whose representations do not get updated during message passing as they do not occur as destination type in any edge type. This may lead to unexpected behavior.
  self.conv = HeteroConv(


In [9]:
# model, y_means, y_stds, training_history, best_val_loss, best_epoch = fit(
#     model, train_loader, val_loader,
#     epochs=EPOCHS,
#     criterion=nn.MSELoss(),
#     optimizer=optimizer,
#     device=device,
#     patience=PATIENCE,
#     evaluation_metric="val_loss",
#     mode="min",
#     verbose=VERBOSITY,
#     experiment_name="cdfg_attn_self_loop_rgcn_experiment",
#     checkpoint_dir="/kaggle/working/artifacts/models",
# )

In [10]:
import json
import os
import subprocess

ARTIFACTS_DIR = "/kaggle/working/artifacts"
CHECKPOINT_DIR = f"{ARTIFACTS_DIR}/models"
RESULTS_DIR = ARTIFACTS_DIR
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

train_config = {
    "tensor_dir": tensor_path,
    "max_per_kernel_type": max_per_type_train_val,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "patience": PATIENCE,
    "verbose": VERBOSITY,
    "seed": SEED,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "lr": 1e-3,
    "experiment_name": "trans_vs_ind_fit",
    "checkpoint_dir": CHECKPOINT_DIR,
    "results_dir": RESULTS_DIR,
}

config_path = "/kaggle/working/train_config.json"
with open(config_path, "w") as f:
    json.dump(train_config, f)

n_gpus = torch.cuda.device_count()
repo_dir = "/kaggle/working/ll-hls4ml"
script = f"{repo_dir}/scripts/train_transductive.py"

if n_gpus >= 2:
    cmd = ["torchrun", f"--nproc_per_node={n_gpus}", script, "--config", config_path]
else:
    cmd = ["python", script, "--config", config_path]

print(f"Launching: {' '.join(cmd)}")
env = os.environ.copy()
env["PYTHONPATH"] = f"{repo_dir}/src" + os.pathsep + env.get("PYTHONPATH", "")
result = subprocess.run(cmd, env=env, cwd=repo_dir)
if result.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {result.returncode}")

training_history_path = f"{RESULTS_DIR}/training_histories.json"
with open(training_history_path) as f:
    training_history = json.load(f)
print(f"Loaded training histories from {training_history_path}")

*********** "exemplar" as inductive set ***********
  Train size: 783,   Val size: 195,   Test size: 300
  instruction: 5239811 nodes, 0 OOV nodes (0.00%)
  variable: 4947994 nodes, 0 OOV nodes (0.00%)
  constant: 49613 nodes, 0 OOV nodes (0.00%)
Training 200 epochs...


Epoch   1/200:   0%|          | 0/123 [00:00<?, ?batch/s]

Epoch   2/200:   0%|          | 0/123 [00:00<?, ?batch/s]

Epoch   3/200:   0%|          | 0/123 [00:00<?, ?batch/s]

Epoch   4/200:   0%|          | 0/123 [00:00<?, ?batch/s]

...

Model saved to /kaggle/working/artifacts/models/trans_vs_ind_fit_exemplar_backup.pt


Epoch 141/200:   0%|          | 0/123 [00:00<?, ?batch/s]

Epoch 142/200:   0%|          | 0/123 [00:00<?, ?batch/s]

Early stopping triggered after 142 epochs.
Best model restored from epoch 117 with val_loss 0.1868
Evaluating on test set...


AttributeError: 'str' object has no attribute 'parent'

## Visualization

In [ ]:
# training_history is written by scripts/train_transductive.py (rank 0)
# Reload here for downstream visualization cells.
import json
with open("/kaggle/working/artifacts/training_histories.json") as f:
    training_history = json.load(f)